# 3 — Most Improved candidate investigations

**In brief**
- **Question:** When a name on the live board looked wrong, what did the evidence say, and what did it change in the model?
- **Answer:** Each case started from a name on the board and ended with a test across past seasons. Voters pass over players who are already stars (none of 26 winners had been an All-Star) and players back from a lost season (none of 26), so both now count against a candidate; an elite level does too, more modestly. An open role, a 20-point scorer leaving, marks winners far more than changing teams. The blip test, written in advance, matches the full season 86% of the time; Alexander-Walker has kept 110% of his leap so far.
- **Decisions** ([decision log](../docs/DECISIONS.md)): [D29](../docs/DECISIONS.md), [D33](../docs/DECISIONS.md), [D38](../docs/DECISIONS.md), [D39](../docs/DECISIONS.md), [D42](../docs/DECISIONS.md), [D43](../docs/DECISIONS.md).
- **Where it fits:** Between the basic race (segment 1) and the revised race (segment 4). Each case ends with what it changed; segment 4 scores every change together.
- **Dated:** written 13 November 2026 from notebooks 14, 16, 17, 18 and 21 (now in [archive/](archive)). Live-season numbers are pinned to the day the case was investigated, so they cannot drift.

**Four case files, each in the same shape:** who raised the question, the evidence, the test and the decision, and what it changed on the board.

| Case | Raised by | Decision |
|---|---|---|
| 3A Did the leap last? | Nickeil Alexander-Walker, last season's winner | The blip test, written in advance ([D33](../docs/DECISIONS.md)) |
| 3B Is the lead believable? | Ty Jerome, 26% on 8 November | "Back from a lost season" counts against a candidate ([D39](../docs/DECISIONS.md)) |
| 3C Already a star? | Alperen Sengun, Victor Wembanyama, Cooper Flagg | "Already an All-Star" ([D38](../docs/DECISIONS.md)), then "already clearly a star" ([D42](../docs/DECISIONS.md)) |
| 3D An open role? | Rui Hachimura, top of the preseason list | "A 20-point scorer left" ([D29](../docs/DECISIONS.md)), kept on points ([D43](../docs/DECISIONS.md)) |

**The discipline:** a name that looks wrong raises a question; only a test across past seasons answers it. Each rule for deciding was written before its test ran. **Tests:** *strict* predicts each season from earlier seasons only; *leave-one-out* from all other seasons. "× a blind guess" is how much more chance the model gave the real winner than an even split. The quicker tests are re-run here; the slow ones (notebooks 17 and 21) appear as dated tables linked to the archived notebook that reproduces them.

In [1]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from unicorn import mip_history as MH
from unicorn.allstars import add_prior_allstar, load_selections
from unicorn.labels import MIP_WINNERS
from unicorn.live_race import LIVE, live_games
from unicorn.mip import ChoiceModel
from unicorn.mip_eval import current_frame, scorecard, winner_ranks
from unicorn.plotting import BLUE, FIG_DIR, INK, INK2, MUTED, ORANGE, SURFACE, apply_style
from unicorn.raw import PROJECT_ROOT

warnings.filterwarnings("ignore")
apply_style()
pd.set_option("display.max_colwidth", None)
DATA = PROJECT_ROOT / "data" / "processed"
sel = load_selections()
g = MH.load_games(); info = MH.player_info(); votes, winners = MH.voting(g)          # 1999-00 → 2025-26, for the 2000-26 tests
live = live_games(refresh=False); live = live[live["season"] == LIVE]                    # the 2026-27 season as cached
board = pd.read_parquet(DATA / "mip_race_daily_2026_27.parquet")                         # the live race, as saved each day
BOARD_DAY = 22                                                                            # 11 November 2026: the board the cases refer to
on_board = lambda name: board[(board["day"] == BOARD_DAY) & (board["player_name"] == name)][["rank", "chance"]].iloc[0]
vr = votes.dropna(subset=["player_id"]).assign(player_id=lambda v: v["player_id"].astype(int)).set_index(["season", "player_id"])["vote_rank"]
print(f"{len(g):,} player-games 1999-00 → 2025-26; {len(winners)} winners; live board on day {BOARD_DAY} "
      f"({board.loc[board['day'] == BOARD_DAY, 'date'].iloc[0]:%d %B})")

673,732 player-games 1999-00 → 2025-26; 26 winners; live board on day 22 (11 November)


## 3A — Nickeil Alexander-Walker: did his leap last?

**Raised by:** last season's winner went from 9.4 to 20.8 points a game. Was it real, or a one-season spike? The question kept coming up, so the answer was fixed in advance, on 30 October, before any of this season was known ([D33](../docs/DECISIONS.md)).

**The blip test.** Share of the leap kept = (points a game now − points before the leap) ÷ (points in the leap season − points before). Judged at the All-Star break of the following season, with at least 25 games:
- **real:** 80% or more kept; **partly real:** 50–80%; **blip:** under 50%.

**Does the rule work?** It was backtested on every big leap since 2000-01: every Most Improved vote-getter who jumped 3+ points, and everyone who jumped 5+. Is the verdict at the break the same as over the whole season? And what happens two seasons on? Game score, the all-round measure, is shown alongside; it does not decide ([D46](../docs/DECISIONS.md)).

In [2]:
from unicorn.leaps import leap_table, verdict

L = leap_table(g, votes, winners)
both = L[(L["verdict at the break"] != "no verdict") & (L["verdict next season"] != "no verdict")]
gboth = L[(L["game-score verdict at the break"] != "no verdict") & (L["game-score verdict next season"] != "no verdict")]
two = L[(L["verdict at the break"] != "no verdict") & (L["verdict two seasons on"] != "no verdict")]
blips, reals = two[two["verdict at the break"] == "blip"], two[two["verdict at the break"] == "real"]
print(f"{len(L)} leaps, 2000-01 → 2024-25")
pd.DataFrame({
    "verdict at the break = verdict over the full season": [f"{(both['verdict at the break'] == both['verdict next season']).mean():.0%} of {len(both)}",
                                                            f"{(gboth['game-score verdict at the break'] == gboth['game-score verdict next season']).mean():.0%} of {len(gboth)}"],
    "'real' at the break became a 'blip'": [int(((both['verdict at the break'] == 'real') & (both['verdict next season'] == 'blip')).sum()),
                                            int(((gboth['game-score verdict at the break'] == 'real') & (gboth['game-score verdict next season'] == 'blip')).sum())]},
    index=["on points (the rule)", "on game score (alongside)"])

609 leaps, 2000-01 → 2024-25


,verdict at the break = verdict over the full season,'real' at the break became a 'blip'
on points (the rule),86% of 541,0
on game score (alongside),83% of 541,1


In [3]:
print(f"two seasons on: {((blips['verdict two seasons on'] == 'real').mean()):.0%} of blips came back to real ({int((blips['verdict two seasons on'] == 'real').sum())} of {len(blips)}); "
      f"{((reals['verdict two seasons on'] == 'blip').mean()):.0%} of reals collapsed ({int((reals['verdict two seasons on'] == 'blip').sum())} of {len(reals)})")
print("past winners, the season after:", L[L["winner"]]["verdict next season"].value_counts().to_dict())

naw = MIP_WINNERS["2025-26"]
mine = live[(live["player_id"] == naw) & (live["day"] <= BOARD_DAY)]           # games up to 11 November (day 22 after opening night)
gs_season = g[g["player_id"] == naw].groupby("season_start")["game_score"].mean()
pts_before, pts_won = g[(g["player_id"] == naw)].groupby("season_start")["pts"].mean().loc[[2024, 2025]]
gs_before, gs_won = gs_season.loc[[2024, 2025]]
share_pts = (mine["pts"].mean() - pts_before) / (pts_won - pts_before)
share_gs = (mine["game_score"].mean() - gs_before) / (gs_won - gs_before)
print(f"Alexander-Walker to 11 November: {len(mine)} games, {mine['pts'].mean():.1f} points ({pts_before:.1f} → {pts_won:.1f} → now), {share_pts:.0%} of the leap kept; "
      f"game score {gs_before:.1f} → {gs_won:.1f} → {mine['game_score'].mean():.1f}, {share_gs:.0%} kept (alongside); verdict: {verdict(share_pts, len(mine))}")

two seasons on: 22% of blips came back to real (25 of 113); 17% of reals collapsed (48 of 287)
past winners, the season after: {'real': 15, 'partly real': 6, 'blip': 4}
Alexander-Walker to 11 November: 12 games, 21.9 points (9.4 → 20.8 → now), 110% of the leap kept; game score 7.5 → 15.3 → 14.6, 91% kept (alongside); verdict: no verdict


**Decision and status ([D33](../docs/DECISIONS.md), [D46](../docs/DECISIONS.md)).** The rule stands as written. The final verdict comes on the first run after the 2026-27 All-Star break, in the live race (notebook 12).

## 3B — Ty Jerome: is his lead believable?

**Raised by:** Ty Jerome led the live race with about a 26% chance on 8 November (day 19 after opening night): 26.6 points a game in 8 games, against 12.5 in his baseline season, at 29.

**The evidence:** every player since 2000-01 who looked like him at the same point of the season, and how his season ended. The strict lookalikes: aged 27+, up 8+ points a game at day 19, from a bench role (24 minutes or fewer) in the baseline season. Their model chances are leave-one-out, from the live race's inputs as they stood then.

In [4]:
DAY = 19
e = add_prior_allstar(MH.in_season_frame(g, info, winners, day=DAY), sel)
e = e[e["games"] >= 5].copy()
ss = g.groupby(["player_id", "season_start"]).agg(gp=("game_id", "size"), ppg=("pts", "mean"))
season_len = g.drop_duplicates(["team_id", "game_id"]).groupby("season_start").apply(lambda d: d.groupby("team_id")["game_id"].nunique().max())
key = list(zip(e["player_id"], e["season_start"]))
e["final_ppg"] = [ss["ppg"].get(k, np.nan) for k in key]
e["share_of_games"] = [ss["gp"].get(k, np.nan) for k in key] / e["season_start"].map(season_len)
e["early_jump"], e["final_jump"] = e["ppg"] - e["base_ppg"], e["final_ppg"] - e["base_ppg"]
e["kept"] = e["final_jump"] / e["early_jump"]
e["vote_rank"] = [vr.get((s, int(p)), np.nan) for s, p in zip(e["season"], e["player_id"])]
FEATS = MH.IN_SEASON_FEATURES + ["over_turn", "prior_allstar"]           # the live race's inputs on 8 November (turning point at 20 points)
d = MH.with_turn(e, "base_ppg", 20)
for s in d["season"].unique():
    te = d["season"] == s
    p = ChoiceModel(FEATS, MH.L2).fit(d[~te]).predict_proba(d[te])
    e.loc[te, "chance"], e.loc[te, "rank"] = p, p.rank(ascending=False, method="min")

def group(x):
    return {"players": len(x), "jump kept by season's end (median)": x["kept"].median(), "played 70%+ of games": (x["share_of_games"] >= 0.7).mean(),
            "got votes": int(x["vote_rank"].notna().sum()), "top 3": int((x["vote_rank"] <= 3).sum()), "won": int(x["mip"].sum()),
            "highest model chance": x["chance"].max()}
hot = e["early_jump"] >= 8
pd.DataFrame({"aged 27+, bench role (the strict lookalikes)": group(e[hot & (e["age"] >= 27) & (e["base_mpg"] <= 24)]),
              "under 27, bench role": group(e[hot & (e["age"] < 27) & (e["base_mpg"] <= 24)]),
              "aged 27+, any role": group(e[hot & (e["age"] >= 27)])}).T.style.format(
    {"players": "{:.0f}", "got votes": "{:.0f}", "top 3": "{:.0f}", "won": "{:.0f}", "jump kept by season's end (median)": "{:.0%}", "played 70%+ of games": "{:.0%}", "highest model chance": "{:.0%}"})

,players,jump kept by season's end (median),played 70%+ of games,got votes,top 3,won,highest model chance
"aged 27+, bench role (the strict lookalikes)",14,65%,64%,3,0,0,9%
"under 27, bench role",53,79%,79%,25,6,3,38%
"aged 27+, any role",31,58%,74%,7,1,1,37%


In [5]:
older = e[hot & (e["age"] >= 27)].nlargest(3, "early_jump")[["season", "player_name", "age", "early_jump"]]
leaders = e[e["rank"] == 1]
print("biggest day-19 jumps at 27 or older:", "; ".join(f"{r.player_name} {r.season} (+{r.early_jump:.1f})" for r in older.itertuples()))
print(f"day-19 leaders: average chance {leaders['chance'].mean():.0%}, won {int(leaders['mip'].sum())} of {len(leaders)}; "
      f"leaders aged 27+: {int((leaders['age'] >= 27).sum())}, won {int(leaders.loc[leaders['age'] >= 27, 'mip'].sum())}")
tj = live[(live["player_name"] == "Ty Jerome") & (live["day"] <= DAY)]
hist_tj = g[g["player_name"] == "Ty Jerome"].groupby("season").agg(games=("game_id", "size"), points=("pts", "mean"))
print(f"Jerome to day {DAY}: {len(tj)} games, {tj['pts'].mean():.1f} points (without his 45-point opener: {tj['pts'].iloc[1:].mean():.1f}); "
      f"his last two seasons: " + "; ".join(f"{s} {int(r.games)} games, {r.points:.1f} points" for s, r in hist_tj.tail(2).iterrows()))

biggest day-19 jumps at 27 or older: Norman Powell 2024-25 (+11.6); AJ Price 2014-15 (+11.4); Brian Grant 2000-01 (+11.0)
day-19 leaders: average chance 19%, won 4 of 26; leaders aged 27+: 3, won 0
Jerome to day 19: 8 games, 26.6 points (without his 45-point opener: 24.0); his last two seasons: 2024-25 70 games, 12.5 points; 2025-26 15 games, 19.7 points


**What stood out:** Jerome's baseline is stale. His 2025-26 season was lost to injury (15 games), so the baseline rule ([D22](../docs/DECISIONS.md)) skips it and compares him with 2024-25. Voters may see a comeback as a *return*, not an improvement.

**The test.** Four candidate inputs were tested at five points of the season (days 14, 19, 30, 60 and 85), in both tests: older hot starts counting for less; **back from a lost season** (the baseline season is older than last season); how much of the jump already showed in that short season; and games played. **The rule:** keep an input only if it improves the fit consistently, not on one day. The winning input is re-run below; the three rejected ones are in the archived [notebook 18C](archive/18_ty_jerome_effect.ipynb).

In [6]:
rows = []
for day in [14, 19, 30, 60, 85]:
    f = current_frame(g, info, winners, sel, day=day, level_col="base_ppg", turn=20)      # the inputs as they stood on 10 November
    for scheme in ["strict", "leave-one-out"]:
        a, b = scorecard(winner_ranks(f, FEATS, scheme=scheme)), scorecard(winner_ranks(f, FEATS + ["stale"], scheme=scheme))
        rows.append({"day": day, "test": scheme, "× a blind guess: without": a["× a blind guess"], "× a blind guess: with": b["× a blind guess"],
                     "winner in top 7: without": a["in top 7"], "winner in top 7: with": b["in top 7"]})
T = pd.DataFrame(rows)
print(f"with 'back from a lost season', the fit improves in {(T['× a blind guess: with'] > T['× a blind guess: without']).sum()} of {len(T)} tests; "
      f"0 of 26 winners had a stale baseline; share of candidates with one: {current_frame(g, info, winners, sel, day=None, level_col='base_ppg', turn=20)['stale'].mean():.1%}")
T.set_index(["day", "test"]).round(2)

with 'back from a lost season', the fit improves in 10 of 10 tests; 0 of 26 winners had a stale baseline; share of candidates with one: 4.6%


× a blind guess: without  × a blind guess: with  \
day test                                                             
14  strict                             8.30                   8.58   
    leave-one-out                      9.30                   9.68   
19  strict                             6.35                   6.52   
    leave-one-out                      7.87                   8.11   
30  strict                             9.01                   9.25   
    leave-one-out                     11.53                  11.92   
60  strict                            14.22                  14.51   
    leave-one-out                     18.72                  19.14   
85  strict                            19.77                  19.94   
    leave-one-out                     25.50                  25.71   

                   winner in top 7: without  winner in top 7: with  
day test                                                            
14  strict                               10                     10  
    leave-one-out                        13                     13  
19  strict                               11                     11  
    leave-one-out                        11                     12  
30  strict                               11                     11  
    leave-one-out                        17                     17  
60  strict                               16                     16  
    leave-one-out                        20                     19  
85  strict                               17                     17  
    leave-one-out                        20                     20

| Input (archived notebook 18C, 10 November) | Tests where the fit improved (of 10) | Result |
|---|---|---|
| Older hot starts count for less (age × early jump) | 0 | Rejected: the age penalty already covers it |
| **Back from a lost season** | **10** | **Adopted ([D39](../docs/DECISIONS.md))** |
| How much of the jump showed in the short season | about half, by tiny amounts | Rejected |
| Share of the team's games played | mixed | Rejected |

**What it changed:** the input gives a player back from a lost season about a fifth of the odds. Jerome fell from 26.5% to 8.3% on day 19 and stayed #1; with the later inputs he was at 12.3% on 11 November. The evidence is thin on its own (0 winners against about 1.2 expected would happen 3 times in 10 by luck); the case rests on the fit improving at every point, in both tests.

## 3C — Already a star? Sengun, Wembanyama and Flagg

**Raised by:** in early November the live board had Alperen Sengun 2nd, with Victor Wembanyama and Scottie Barnes in the top 7. All three were already All-Stars. Later, Wembanyama (a baseline game score of 22) and Cooper Flagg (a 21-point rookie season) still ranked high.

**First question: do voters pass over players who have already been All-Stars?** Only selections from *before* the season count: many winners were first picked in the year they won, so the current season's selection would leak the answer.

In [7]:
VIEWS = {"end of the previous season": (MH.end_of_previous_frame(g, info, winners), MH.PREV_FEATURES, "ppg_last"),
         f"mid-season (day {MH.DAY_MID})": (MH.in_season_frame(g, info, winners, day=MH.DAY_MID), MH.IN_SEASON_FEATURES, "base_ppg"),
         "end of the season": (MH.in_season_frame(g, info, winners), MH.IN_SEASON_FEATURES, "base_ppg")}
rows = []
for name, (f, feats, level) in VIEWS.items():
    f = MH.with_turn(add_prior_allstar(f, sel), level, 20)
    f["vote_rank"] = [vr.get((s, int(p)), np.nan) for s, p in zip(f["season"], f["player_id"])]
    expected = sum(ChoiceModel(feats + ["over_turn"], MH.L2).fit(f[f["season"] != s]).predict_proba(f[f["season"] == s])[f.loc[f["season"] == s, "prior_allstar"] == 1].sum()
                   for s in f["season"].unique())                       # wins the model WITHOUT the input expects them to take (leave-one-out)
    pa = f[f["prior_allstar"] == 1]
    rows.append({"view": name, "share of candidates": len(pa) / len(f), "finished top 5": int((pa["vote_rank"] <= 5).sum()), "won": int(pa["mip"].sum()),
                 "wins expected without the input": expected, "chance of 0 wins if that were right": np.exp(-expected)})
pd.DataFrame(rows).set_index("view").style.format({"share of candidates": "{:.0%}", "wins expected without the input": "{:.1f}", "chance of 0 wins if that were right": "{:.0%}"})

,share of candidates,finished top 5,won,wins expected without the input,chance of 0 wins if that were right
view,,,,,
end of the previous season,15%,5,0,2.5,8%
mid-season (day 85),18%,5,0,2.2,11%
end of the season,17%,5,0,2.2,11%


**The test (archived [notebook 17](archive/17_already_a_star.ipynb), 9 November).** Re-running it takes several minutes, so its results are shown here:

| Test | Without | With "already an All-Star" |
|---|---|---|
| Preseason, strict: winner's median rank | 15 | **7** |
| Preseason, strict: winner in our top 7 | 4 of 9 | **5 of 9** |
| Live race at day 85: winner in our top 7 | 11 of 15 | **12 of 15** |
| 2000-26, mid-season: winner in our top 7 | 16 of 21 | **17 of 21** |

**Decision ([D38](../docs/DECISIONS.md)):** adopted, as a penalty rather than a ban: a prior All-Star has about a quarter of the odds of an otherwise identical player.

**Second question (12 November): is an elite level a discount too, even without an All-Star selection?** The game-score turning point turned out to *neutralise* a high level rather than count against it, so Wembanyama and Flagg could still rank high.

In [8]:
f = add_prior_allstar(MH.in_season_frame(g, info, winners), sel)
f["vote_rank"] = [vr.get((s, int(p)), np.nan) for s, p in zip(f["season"], f["player_id"])]
elite = f[f["base_game_score"] >= 22]
print(f"candidates with a baseline game score of 22+, 2000-01 → 2025-26: {len(elite)}; got votes: {int(elite['vote_rank'].notna().sum())}; "
      f"finished top 5: {int((elite['vote_rank'] <= 5).sum())}; highest baseline game score of any winner: {f.loc[f['mip'], 'base_game_score'].max():.1f}")

candidates with a baseline game score of 22+, 2000-01 → 2025-26: 75; got votes: 5; finished top 5: 0; highest baseline game score of any winner: 14.6


**The test (archived [notebook 21C–21D](archive/21_beyond_ppg.ipynb), 12 November; about 30 minutes to re-run).** Three options, each compared with the model as it stood, on the same seasons (2012-13 → 2025-26, when star scores exist). **The rule, written first:** keep an option only if it beats the current model in the strict test at 2 of 3 points of the season, with no fewer winners in the top 7; if more than one passes, the larger strict gain, then leave-one-out, then the simpler.

| Option | Better at (strict, of 3 points) | Change, strict | Change, leave-one-out |
|---|---|---|---|
| A baseline game score of 16+ | 3 | +4.3% | +3.9% |
| Last season in our star tier (the top 10% by star score) | 2 | +2.4% | +4.5% |
| **Both** | **3** | **+6.4%** | **+7.8%** |

**Decision ([D42](../docs/DECISIONS.md)):** both, each cutting a player's odds to about a third. The evidence is modest: the earlier model already expected only about 0.2 wins from such players.

In [9]:
for name in ["Alperen Sengun", "Scottie Barnes", "Victor Wembanyama", "Cooper Flagg"]:
    r = on_board(name)
    print(f"{name}: #{int(r['rank'])} on 11 November ({r['chance']:.1%})")

Alperen Sengun: #59 on 11 November (0.3%)
Scottie Barnes: #101 on 11 November (0.1%)
Victor Wembanyama: #17 on 11 November (1.4%)
Cooper Flagg: #10 on 11 November (2.1%)


## 3D — An open role? Hachimura and the preseason list

**Raised by:** the preseason list was led by Rui Hachimura, whose team (the Clippers) had lost Kawhi Leonard; six of its top 7 played for a team that had lost a 20-point scorer. Is changing teams what matters, or the role that opens up?

**The evidence** (14 preseasons, 2012-13 → 2025-26; the archived [notebook 14](archive/14_team_changes.ipynb) has 21 seasons and a fuller look at new teammates). "A 20-point scorer left": a player who scored 20+ a game for the team last season was seen with another team in preseason, and no 20-point scorer arrived.

In [10]:
from unicorn.preseason import LEAN, TURN, USED_D29, add_pool_flags, add_star_left, add_turning_point, load_preseason, preseason_frame
regular = pd.read_parquet(DATA / "player_games.parquet"); ps = pd.read_parquet(DATA / "player_season.parquet")
pre = load_preseason(regular=regular, refresh_latest=False)
F = add_star_left(add_turning_point(add_pool_flags(preseason_frame(pre, regular, ps), MIP_WINNERS)), pre, regular)
H = F[F["season"] != LIVE].reset_index(drop=True)
w = H[H["mip"]]
pd.DataFrame({"winners": [w["new_team"].mean(), w["star_left"].mean()], "all candidates": [H["new_team"].mean(), H["star_left"].mean()]},
             index=["changed team", "a 20-point scorer left (none arrived)"]).style.format("{:.0%}")

,winners,all candidates
changed team,36%,34%
a 20-point scorer left (none arrived),43%,8%


**The test (re-run).** Which version of the input, added to the preseason model as it stood (preseason form and the turning point)? **The rule:** keep the simplest version that does better in both tests, and reject any version whose shape has no basketball sense.

In [11]:
pre_end = regular.sort_values("game_date").groupby(["player_id", "season_start"]).agg(team=("team_id", "last"), ppg=("pts", "mean")).reset_index()
pre_team = pre.sort_values("game_date").groupby(["player_id", "season_start"])["team_id"].last()
pre_end["next"] = [pre_team.get((p, s + 1)) for p, s in zip(pre_end["player_id"], pre_end["season_start"])]
gone = pre_end[pre_end["next"].notna() & (pre_end["next"] != pre_end["team"])]
best_gone = gone.groupby(["team", gone["season_start"] + 1])["ppg"].max()
H["top_scorer_gone"] = [best_gone.get(k, 0) for k in zip(H["team_id"], H["season_start"])]
H["star_left_ppg"] = np.where(H["top_scorer_gone"] >= 20, H["top_scorer_gone"] - 19, 0.0)
BANDS = []
for lo, hi in [(20, 22), (22, 25), (25, 27), (27, 99)]:
    H[f"band_{lo}"] = ((H["top_scorer_gone"] >= lo) & (H["top_scorer_gone"] < hi)).astype(float); BANDS.append(f"band_{lo}")
variants = {"no 'scorer left' input": LEAN + TURN, "a 20+ scorer left": LEAN + TURN + ["star_left_any"],
            "a 20+ scorer left and none arrived (chosen)": USED_D29, "four bands (20-22, 22-25, 25-27, 27+)": LEAN + TURN + BANDS,
            "graded by the departed scorer's points": LEAN + TURN + ["star_left_ppg"]}
cmp = pd.DataFrame({k: {**{f"leave-one-out: {a}": b for a, b in scorecard(winner_ranks(H, f, 3.0, "loso")).items() if a in ("in top 7", "× a blind guess")},
                        **{f"strict: {a}": b for a, b in scorecard(winner_ranks(H, f, 3.0, "strict")).items() if a in ("in top 7", "× a blind guess")}}
                    for k, f in variants.items()}).T
print("band weights:", ChoiceModel(LEAN + TURN + BANDS, 3.0).fit(H).coefficients()[BANDS].round(2).to_dict(),
      "| winners in each band:", {b: int(H.loc[H["mip"], b].sum()) for b in BANDS})
cmp.round(1)

band weights: {'band_20': -0.21, 'band_22': 0.54, 'band_25': 0.37, 'band_27': -0.16} | winners in each band: {'band_20': 0, 'band_22': 4, 'band_25': 2, 'band_27': 0}


,leave-one-out: in top 7,leave-one-out: × a blind guess,strict: in top 7,strict: × a blind guess
no 'scorer left' input,3.0,5.6,1.0,3.3
a 20+ scorer left,7.0,6.8,3.0,4.6
a 20+ scorer left and none arrived (chosen),7.0,7.1,4.0,4.9
"four bands (20-22, 22-25, 25-27, 27+)",8.0,9.4,3.0,5.2
graded by the departed scorer's points,5.0,6.6,3.0,4.2


**Decision ([D29](../docs/DECISIONS.md)): yes/no, "a 20-point scorer left and none arrived".** The four bands score best on paper but memorise: losing a 21- or 28-point scorer "hurts" and a 23-point scorer "helps", a shape with no basketball sense, fitted to six winners.

**Points or an all-round measure? (archived [notebook 21E](archive/21_beyond_ppg.ipynb), 12 November).** Each measure's threshold chosen inside training; **the rule, written first:** replace points only if better in the strict test, not worse in leave-one-out, and no fewer winners in the top 7 in either.

| Measure | Strict | Leave-one-out | Winner in top 7 (strict / leave-one-out) |
|---|---|---|---|
| **Points** | 6.1× | 10.7× | 4 / 9 |
| Game score | 6.2× (+2%) | 9.8× (−8.5%) | 5 / 10 |
| Usage | 4.2× (−32%) | 7.4× (−30%) | 3 / 6 |

**Decision ([D43](../docs/DECISIONS.md)): points stay.** What opens a role is the departed player's *scoring*.

In [12]:
for name in ["Rui Hachimura", "Brandon Miller", "Cooper Flagg"]:
    r = on_board(name)
    print(f"{name}: #{int(r['rank'])} on the live board on 11 November ({r['chance']:.1%})")

Rui Hachimura: #35 on the live board on 11 November (0.6%)
Brandon Miller: #42 on the live board on 11 November (0.5%)
Cooper Flagg: #10 on the live board on 11 November (2.1%)


## Findings (13 November 2026)

**3A Alexander-Walker: the blip test works, within its horizon.**
- On 541 past leaps, the verdict at the All-Star break matched the full season 86% of the time, and "real" at the break never became a "blip". On game score, shown alongside, 83%.
- It looks one season ahead only: two seasons on, 22% of blips had come back (25 of 113; Julius Randle is the best-known) and 17% of real leaps had collapsed (48 of 287, mostly through injury).
- **Alexander-Walker to 11 November:** 21.9 points a game in 12 games, 110% of his leap kept (91% on game score). On track for "real"; the verdict comes at the break.

**3B Ty Jerome: a real candidate, an unlikely winner.**
- No player aged 27 or older had made a day-19 jump this large; the next biggest were Norman Powell (+11.6) and AJ Price (+11.4).
- The 14 strict lookalikes kept a median 65% of their jump; none finished top 3, none won. Day-19 leaders won 4 times in 26, and the three aged 27+ never.
- **"Back from a lost season" improved the fit in all 10 tests** (five points of the season, both tests) and was adopted ([D39](../docs/DECISIONS.md)). 0 of 26 winners were in that position, against 4.6% of candidates. Jerome fell from 26.5% to 8.3% at day 19, still #1.

**3C Already a star: a discount, not a ban.**
- Prior All-Stars are 15–18% of candidates and won 0 of 26 awards, where the model without the input expected 2.2–2.5 (zero wins about 1 time in 10 by luck). The input improved every test and was adopted ([D38](../docs/DECISIONS.md)).
- 75 candidates since 2000-01 had a baseline game score of 22+; none finished top 5, and no winner's baseline was above 14.6. A baseline game score of 16+ and last season in our star tier, together, sharpened the strict test by 6.4% ([D42](../docs/DECISIONS.md)); the evidence is modest.
- **On 11 November:** Sengun #59, Barnes #101, Wembanyama #17, Flagg #10.

**3D An open role, not a new team.**
- Over 14 preseasons, winners changed teams no more often than other candidates (36% against 34%), but **43% had a 20-point scorer leave their team, against 8% of candidates.**
- The simple yes/no version was kept ([D29](../docs/DECISIONS.md)); finer scoring bands fitted better on paper but learned a shape with no basketball sense. Points beat game score and usage as the measure of the departed star ([D43](../docs/DECISIONS.md)).
- **On 11 November:** Hachimura, #1 before the season, is #35; Brandon Miller #42. The clue finds players set up to improve, not the winner.

**Across the four cases:** three changed the model (D38, D39, D42) and two confirmed a rule (D29 with D43, and D33). Three of those changes were prompted by names on the board, which is why the model is now frozen for the season ([D44](../docs/DECISIONS.md)): 2026-27 is its first clean test.